# v2 - weather forecasts, ensembles, exports, probabilistic forecasts

**Runtime → Run all.** About 100 minutes on Colab's CPU runtime. Nothing to decide
along the way. The design is fixed in advance in `docs/spec-v2.md`.

What it does, in order:

1. Rebuilds the demand data from IESO
2. Runs the tests - the leakage guards for features, the probabilistic baseline and the
   weather-forecast rule. **A failure stops the notebook.**
3. Downloads archived GFS temperature forecasts from Open-Meteo and **verifies** that each
   archived value comes from the model run the legality rule assumes. **If verification
   fails, the notebook stops** - Track C would otherwise be using forecasts that were not
   yet published at the 12:30 origin.
4. Runs `python -m src.evaluate.run_v2`: development window 2023 (threshold and ensemble
   weights), then the clean holdout (2026-01-01 to 2026-06-13), the official archive and
   the seen v1 windows
5. Runs `scripts/diagnose_v2.py`, post-hoc checks on Track B and Track C that read the
   predictions just made (exploratory, not part of the pre-registered results)
6. Shows the headline and hands you a zip of `reports/v2/`

Progress prints a line per model per window. Long silences are a single model fitting.

If Colab disconnects, run all again: predictions already computed in this session are cached
and skipped. With `USE_DRIVE = True` the cache also survives a full runtime restart.

## 1. Setup

In [ ]:
# polars is upgraded explicitly: pip leaves an older preinstalled copy in place otherwise.
!pip install -q -U polars
!pip install -q duckdb holidays lightgbm statsforecast pytest

In [ ]:
USE_DRIVE = False   # True keeps downloads and the prediction cache in Drive (asks for Drive access)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os, pathlib, subprocess, shutil

REPO_URL = "https://github.com/ethantosc/ieso-demand-forecast.git"
REPO_DIR = pathlib.Path("/content/ieso-demand-forecast")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

Only rebuildable directories are redirected. `data/raw/forecasts/` - the harvested IESO
forecast archive - stays where the clone put it.

In [ ]:
DATA_ROOT = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data" if USE_DRIVE
                         else "/content/ieso-demand-forecast-data")
for sub in ("raw/demand", "raw/weather", "raw/weather_fc", "staging", "curated"):
    (DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)
    link = pathlib.Path("data") / sub
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if [p for p in link.rglob("*") if p.suffix.lower() in {".csv", ".parquet", ".json"}]:
            raise SystemExit(f"data/{sub} holds data files. Decide before re-running.")
        shutil.rmtree(link)
    link.symlink_to(DATA_ROOT / sub)
print("store:", DATA_ROOT, "| IESO forecast archive files:", len(list(pathlib.Path("data/raw/forecasts").rglob("*.xml"))))

In [ ]:
subprocess.run(["python", "-m", "src.ingest.fetch_demand"], check=True)
subprocess.run(["python", "-m", "src.ingest.demand"], check=True)

## 2. Tests - a failure stops here

In [ ]:
result = subprocess.run(["python", "-m", "pytest", "tests", "-q"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit("Tests failed - do not trust any result below. Send the output above.")

## 3. Weather forecasts - download and verify

The verification prints, for every archived forecast value it checks, the lead time of the
model run that produced it. The legality rule needs leads of **24-29 h for day1 and 48-53 h
for day2**. If at least 95% match, the rule holds. If not, the notebook stops.

In [ ]:
fc = subprocess.run(["python", "-m", "src.ingest.fetch_weather_forecast"], capture_output=True, text=True)
print(fc.stdout[-6000:], fc.stderr[-3000:])
if fc.returncode == 2:
    raise SystemExit("Forecast verification FAILED - stopping before any model runs. Send the output above.")
if fc.returncode != 0:
    print("Weather forecasts unavailable - Track C, its ensembles and controls will be skipped; "
          "everything else still runs.")

## 4. Run v2

In [ ]:
!python -m src.evaluate.run_v2

Post-hoc diagnostics. They read the prediction cache above, so they must run in the same
session. A failure here does not affect the results.

In [ ]:
!python scripts/diagnose_v2.py | tail -2

## 5. Headline

In [ ]:
from IPython.display import Image, Markdown, display

report = pathlib.Path("reports/v2/findings_auto.md").read_text()
start = report.index("## 1. Headline")
end = report.index("\n## ", start + 5)
display(Markdown(report[start:end]))
for name in ["holdout_ladder", "holdout_by_hour", "holdout_reliability", "holdout_exports_by_hour", "prospective_daily"]:
    path = pathlib.Path(f"reports/v2/figures/{name}.png")
    if path.exists():
        display(Image(filename=str(path)))

## 6. Take the outputs with you

Downloads `ieso_v2_results.zip`. Send it to Claude, which commits it and writes the v2
section of `findings.md`.

**Rerunning later** adds prospective days - the clean, growing comparison against IESO,
including this coming winter. Any later change to model code resets that window
automatically, so reruns only count if the code is untouched.

In [ ]:
out = pathlib.Path("/content/ieso_v2_results")
if out.exists():
    shutil.rmtree(out)
shutil.copytree("reports/v2", out / "reports" / "v2")
for extra in ("weather_fc_meta.json", "weather_fc_coverage.csv", "weather_coverage.csv"):
    src = pathlib.Path("data/staging") / extra
    if src.exists():
        shutil.copy(src, out / "reports" / "v2" / extra)
archive = shutil.make_archive("/content/ieso_v2_results", "zip", out)
if USE_DRIVE:
    shutil.copy(archive, DATA_ROOT / "ieso_v2_results.zip")
from google.colab import files
files.download(archive)